In [6]:
import os
import numpy as np
import pandas as pd
import networkx as nx
import statsmodels.api as sm
from scipy.stats import spearmanr
from sklearn.neighbors import kneighbors_graph
from tqdm import tqdm

DATA_DIR = "../network_dgree/Zhang_data"
MEMBER_DIR = f"{DATA_DIR}/hunt24"
OUTPUT_PATH = "network_msr_results_871.csv"

K_NN = 5                 
MASS_WINDOWS = (5, 10, 20) 
N_BOUNDS = (25, 1500)      
PROB_MIN = 0.8           
REQ_COLS = ['prob', 'mass50', 'radeg', 'dedeg', 'plx', 'e_plx']


clean_cols = lambda df: df.rename(columns=lambda c: c.lower().strip())
make_key = lambda s: s.str.lower().str.replace(' ', '', regex=False)
df_meta = clean_cols(pd.read_csv(f"{DATA_DIR}/hunt24.csv"))
df_tau = clean_cols(pd.read_csv(f"{DATA_DIR}/Cluster_Analysis_Toolkit/Cluster_MSR_Tau_Data.csv"))
df_meta['key'] = make_key(df_meta['name'])
df_tau['key'] = make_key(df_tau['name'])
meta_dict = df_meta.set_index('key').to_dict('index')


def get_local_3d_coords(m, p_cl, s_cl):
    ra, dec = np.radians(m['radeg'].values), np.radians(m['dedeg'].values)
    s_star2, s_cl2 = m['e_plx'].values ** 2, max(s_cl, 0.01) ** 2
    
    pc = (m['plx'].values * s_cl2 + p_cl * s_star2) / (s_cl2 + s_star2)
    d = 1000.0 / pc
    xyz = np.column_stack([d * np.cos(dec) * np.cos(ra),
                           d * np.cos(dec) * np.sin(ra),
                           d * np.sin(dec)])
    return xyz - xyz.mean(axis=0)

results = []
for _, row in tqdm(df_tau.iterrows(), total=len(df_tau), desc="Extracting Network MSR"):
    key = row['key']
    fpath = f"{MEMBER_DIR}/{key}.csv"
    if not os.path.exists(fpath) or key not in meta_dict:
        continue
    m = clean_cols(pd.read_csv(fpath)).dropna(subset=REQ_COLS)
    m = m[m['prob'] >= PROB_MIN]
    if not (N_BOUNDS[0] <= len(m) <= N_BOUNDS[1]):
        continue
    info = meta_dict[key]
    xyz = get_local_3d_coords(m, info['plx'], info.get('e_plx', 0.05))
    
    A = kneighbors_graph(xyz, n_neighbors=K_NN, mode='distance', include_self=False)
    G = nx.from_scipy_sparse_array(A)
    dc = np.fromiter(nx.degree_centrality(G).values(), dtype=float)
    cl = np.fromiter(nx.closeness_centrality(G).values(), dtype=float)
    bc = np.fromiter(nx.betweenness_centrality(G, weight='weight').values(), dtype=float)
    
    norm_metrics = {
        'dc': dc / np.mean(dc),
        'cl': cl / np.mean(cl),
        'bc': bc / (np.mean(bc) + 1e-9)
    }

    res_row = {
        'Name': row['name'],
        'N_members': len(m),
        'lambda3d_5': row.get('lambda3d_5', np.nan),
        'tau': row.get('tau', np.nan)
    }
    
    mass_sorted_idx = np.argsort(m['mass50'].values)
    for n_m in MASS_WINDOWS:
        idx = mass_sorted_idx[-n_m:]
        for prefix, arr in norm_metrics.items():
            res_row[f'{prefix}_msr_{n_m}'] = np.mean(arr[idx])
            
    results.append(res_row)

df_res = pd.DataFrame(results).dropna(subset=['cl_msr_10', 'lambda3d_5', 'tau'])
df_res = df_res[df_res['tau'] > 0].copy()
df_res['log_tau'] = np.log10(df_res['tau'])

X_ols = sm.add_constant(df_res['lambda3d_5'])
for n_m in MASS_WINDOWS:
    col_cl = f'cl_msr_{n_m}'
    if col_cl in df_res.columns:
        df_res[f'cl_residual_{n_m}'] = sm.OLS(df_res[col_cl], X_ols).fit().resid

df_res.to_csv(OUTPUT_PATH, index=False)


Extracting Network MSR: 100%|██████████| 871/871 [00:41<00:00, 21.03it/s]
